# Distributed Spam Detection
Krzysztof Kopel, 2026

In this notebook I will try to construct (and later, deploy) an ML model to distinguish spam and normal emails. The model will be created in Ray, a library supporting distributed computing.

In [1]:
import ray
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Only legacy Keras works well with Ray

In [2]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-06-01 02:01:33,024	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


## Data loading and preprocessing

I will use Enron Spam dataset.

In [3]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-06-01 02:01:50,461	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-06-01 02:01:50,480	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-06-01 02:01:50,513	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-06-01_02-01-23_495763_25608\logs\ray-data
2026-06-01 02:01:50,514	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-06-01 02:01:50,517	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (1.8GiB out of 4.2GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJE

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [4]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

2026-06-01 02:01:51,554	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-06-01 02:01:51,556	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_4_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-06-01_02-01-23_495763_25608\logs\ray-data
2026-06-01 02:01:51,557	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> AllToAllOperator[MapBatches(preprocess_batch)->RandomShuffle]
2026-06-01 02:01:51,563	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-06-01 02:01:51,565	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-01 02:01:51,566	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/917.5MiB object store
2026-06-01 02:01:51,566	INFO logging_progress.py:181 -- 
2026-06-01 02:01:51,566	INFO logging_progress.py:231 -- MapBatches(preprocess_batch)->RandomShuffle: 0/1
2026-06-01 02:01:51,567	INFO logging_progress

Train set:
{'Message ID': 10560, 'Subject': 'do i require an attorney to use this system and clean up my record', 'Message': "calls about late payments are discontinued dead in their tracks .\nwe have pioneered an advanced system of proven strategies\nthat will get the creditors and debt collectors off your back for good\nour debt termination program has legally stopped millions of dollars worth\nof debt from being collected .\ncheck out our elimination program here\nhttp : / / bxr . km . classypeopleitems . com / g 8 /\npo box in link above and you can say no thank you for future\nday was now breaking , and several of the tatars appeared and examined the\nbody of the turk with grunts of surprise , for there was no mark upon him to\nshow how he had been slain . supposing him to be dead , they tossed him aside\nand forgot all about him\nrob had secured his ruby ring again , and going to the chief ' s tent he\nshowed the jewel to the guard and was at once admitted", 'Spam/Ham': 1, 'Date'

## Training a model

In [5]:
import tensorflow as tf

def build_model() -> tf.keras.Sequential:

    return tf.keras.Sequential([
        tf.keras.Input(shape=(200,), dtype=tf.int32),
        tf.keras.layers.Embedding(input_dim=10000, output_dim=32),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

In [6]:
from ray.train.tensorflow.keras import ReportCheckpointCallback

def training_loop(config: dict):
    tf.keras.backend.clear_session()
    strategy = tf.distribute.MultiWorkerMirroredStrategy()

    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 4)
    train = ray.train.get_dataset_shard("train").to_tf(feature_columns="Text", label_columns="Spam/Ham", batch_size=batch_size)
    test = ray.train.get_dataset_shard("test").to_tf(feature_columns="Text", label_columns="Spam/Ham", batch_size=batch_size)

    with strategy.scope():
        model = build_model()
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    model.fit(train, validation_data=test, batch_size=batch_size, epochs=epochs, callbacks=[ReportCheckpointCallback()])


In [7]:
global_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
global_vectorizer.adapt(train.to_pandas()["Text"])
vocab = global_vectorizer.get_vocabulary()

def tokenize_batch(batch):
    local_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
    local_vectorizer.set_vocabulary(vocab)
    tokenized_text = local_vectorizer(batch["Text"]).numpy()
    batch["Text"] = tokenized_text
    return batch

train = train.map_batches(tokenize_batch, batch_format="numpy")
test = test.map_batches(tokenize_batch, batch_format="numpy")

2026-06-01 02:01:56,526	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_2
2026-06-01 02:01:56,533	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_2 =======
2026-06-01 02:01:56,536	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-01 02:01:56,536	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/917.5MiB object store
2026-06-01 02:01:56,537	INFO logging_progress.py:192 -- ============================================
2026-06-01 02:01:56,545	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_6_2 execution finished in 0.03 seconds


In [9]:
from ray.train.tensorflow import TensorflowTrainer
from ray.train import RunConfig

scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=False)

trainer = TensorflowTrainer(
    train_loop_per_worker=training_loop,
    train_loop_config={"batch_size": 32, "epochs": 5},
    scaling_config=scaling_config,
    run_config=RunConfig(storage_path=os.path.abspath("./artifacts")),
    datasets={"train": train, "test": test}
)
results = trainer.fit()

(TrainController pid=37140) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=37140) I0000 00:00:1780272226.377707   16880 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=37140) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=37140) I0000 00:00:1780272227.879767   16880 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=37140) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\l

(pid=31196) Running Dataset dataset_25_0.: 0.00 row [00:00, ? row/s]

(pid=31196) - MapBatches(tokenize_batch):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=31196) - limit=1:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=31196) ✔️  Dataset dataset_25_0 execution finished in 1.36 seconds


...

(pid=17096) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=5256) 727/844 [========================>.....] - ETA: 0s - loss: 0.0177 - accuracy: 1.9940 [repeated 8x across cluster]


(RayTrainWorker pid=4408) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/Documents/Programy_Python/DistributedMLProject/artifacts/ray_train_run-2026-06-01_02-03-40/checkpoint_2026-06-01_02-05-18.613757)
(RayTrainWorker pid=4408) Reporting training result 5: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/Documents/Programy_Python/DistributedMLProject/artifacts/ray_train_run-2026-06-01_02-03-40/checkpoint_2026-06-01_02-05-18.613757), metrics={'loss': 0.008730251342058182, 'accuracy': 0.9969968795776367, 'val_loss': 0.029244305565953255, 'val_accuracy': 0.9909549355506897}, validation=False)


(RayTrainWorker pid=4408) 
(RayTrainWorker pid=4408) 844/844 [==============================] - 9s 9ms/step - loss: 0.0087 - accuracy: 0.9970 - val_loss: 0.0292 - val_accuracy: 0.9910
(RayTrainWorker pid=4408) 755/844 [=========================>....] - ETA: 0s - loss: 0.0176 - accuracy: 1.9941 [repeated 8x across cluster]
(RayTrainWorker pid=5256) 781/844 [==========================>...] - ETA: 0s - loss: 0.0175 - accuracy: 1.9942 [repeated 8x across cluster]
(RayTrainWorker pid=5256) 809/844 [===========================>..] - ETA: 0s - loss: 0.0176 - accuracy: 1.9940 [repeated 8x across cluster]
(RayTrainWorker pid=5256) 842/844 [============================>.] - ETA: 0s - loss: 0.0175 - accuracy: 1.9940 [repeated 10x across cluster]
(RayTrainWorker pid=5256) 


(raylet) Stack (most recent call first):
(raylet)   File "<frozen importlib._bootstrap_external>", line 757 in _compile_bytecode
(raylet)   File "<frozen importlib._bootstrap_external>", line 1128 in get_code
(raylet)   File "<frozen importlib._bootstrap_external>", line 995 in exec_module
(raylet)   File "<frozen importlib._bootstrap>", line 935 in _load_unlocked
(raylet)   File "<frozen importlib._bootstrap>", line 1331 in _find_and_load_unlocked
(raylet)   File "<frozen importlib._bootstrap>", line 1360 in _find_and_load
(raylet)   File "C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\experimental\channel\accelerator_context.py", line 8 in <module>
(raylet)   File "<frozen importlib._bootstrap>", line 488 in _call_with_frames_removed
(raylet)   File "C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\experimental\channel\common.py", line 21 in <module>
(raylet)   File "C:\Users\krzys\Documents\Programy_

### Evaluation

In [10]:
print(f"Final results: {results.metrics}")

Final results: {'loss': 0.008730251342058182, 'accuracy': 0.9969968795776367, 'val_loss': 0.029244305565953255, 'val_accuracy': 0.9909549355506897}


Model seems to be working correctly and is ready for deployment.

In [11]:
with open("artifacts/vocabulary.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(global_vectorizer.get_vocabulary()))